In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import EfficientNetB0
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
val_dir = val_dir + test_dir


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras import Model

# Paths
train_dir = '/content/drive/MyDrive/Antika data_augmented/Train'
val_dir   = '/content/drive/MyDrive/Antika data/Valid'

# Parameters
IMG_SIZE = (224, 224)   # ✔ Better for EfficientNet
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Load datasets using tf.data
train_ds = image_dataset_from_directory(
    train_dir,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = image_dataset_from_directory(
    val_dir,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Smoother pipeline (3x faster)
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

# STRONG Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.3),
])

# Build EfficientNetB0 base model
base_model = EfficientNetB0(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # Stage 1: freeze

# Build model
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
outputs = Dense(6, activation="softmax")(x)

model = Model(inputs, outputs)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ---- Stage 1 Training ----
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

# ---- Stage 2 Fine-tuning (Unfreeze last 60 layers) ----
base_model.trainable = True

for layer in base_model.layers[:-60]:
    layer.trainable = False  # freeze everything except last 60

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_finetune = model.fit(train_ds, validation_data=val_ds, epochs=10)



Found 4453 files belonging to 6 classes.
Found 96 files belonging to 6 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 453s 3s/step - accuracy: 0.4929 - loss: 1.3195 - val_accuracy: 0.4688 - val_loss: 1.2781
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 398s 3s/step - accuracy: 0.7513 - loss: 0.7114 - val_accuracy: 0.5104 - val_loss: 1.1854
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 436s 3s/step - accuracy: 0.8130 - loss: 0.5527 - val_accuracy: 0.5208 - val_loss: 1.1367
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 388s 3s/step - accuracy: 0.8308 - loss: 0.5025 - val_accuracy: 0.5417 - val_loss: 1.1244
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 387s 3s/step - accuracy: 0.8481 - loss: 0.4485 - val_accuracy: 0.5417 - val_loss: 1.1065
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 469s 3s/step - accuracy: 0.8532 - loss: 0.4169 - val_accuracy: 0.5312 - val_loss: 1.1314
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 398s 3s/step - accuracy: 0.8624 - loss: 0.4066 - val_ac

In [ ]:
# Save the model
model.save("/content/drive/MyDrive/Antika_model.keras")
